In [ ]:
# ── Cell 1: Load config and read review_analytics from S3 ────────────────────
exec(open("/Workspace/ecommerce_retail/config.py").read())

df_reviews = spark.read \
    .option("fs.s3a.access.key", ACCESS_KEY) \
    .option("fs.s3a.secret.key", SECRET_KEY) \
    .option("fs.s3a.session.token", SESSION_TOKEN) \
    .option("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.TemporaryAWSCredentialsProvider") \
    .parquet(f"{S3_GOLD_BASE}/review_analytics/")

df_reviews.createOrReplaceTempView("review_analytics")
print(f"✅ {df_reviews.count()} rows loaded")

In [ ]:
# ── Cell 2: Install libraries ─────────────────────────────────────────────────
%pip install plotly pandas

In [ ]:
# ── Cell 3: Imports ───────────────────────────────────────────────────────────
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# ── Cell 4: KPI Summary Table ─────────────────────────────────────────────────
df = spark.sql("""
    SELECT
        COUNT(review_id)                                        AS total_reviews,
        COUNT(DISTINCT product_id)                              AS products_reviewed,
        COUNT(DISTINCT customer_id)                             AS unique_reviewers,
        ROUND(AVG(rating), 2)                                   AS overall_avg_rating,
        SUM(CASE WHEN verified_purchase = true THEN 1 ELSE 0 END) AS verified_reviews,
        SUM(CASE WHEN sentiment = 'Positive' THEN 1 ELSE 0 END) AS positive_reviews,
        SUM(CASE WHEN sentiment = 'Negative' THEN 1 ELSE 0 END) AS negative_reviews,
        SUM(CASE WHEN sentiment = 'Neutral'  THEN 1 ELSE 0 END) AS neutral_reviews,
        SUM(CASE WHEN is_prolific_reviewer = true THEN 1 ELSE 0 END) AS prolific_reviewers
    FROM review_analytics
""").toPandas()

kpi = df.T.reset_index()
kpi.columns = ["KPI", "Value"]

fig = go.Figure(data=[go.Table(
    header=dict(
        values=["<b>KPI</b>", "<b>Value</b>"],
        fill_color="#2E3A59",
        font=dict(color="white", size=13),
        align="left",
        height=35
    ),
    cells=dict(
        values=[kpi["KPI"], kpi["Value"]],
        fill_color=[["#F1EFE8", "#FFFFFF"] * len(kpi)],
        font=dict(size=12),
        align="left",
        height=30
    )
)])
fig.update_layout(title="⭐ Review Analytics — KPI Summary")
fig.show()

In [ ]:
# ── Cell 5: Chart 1 — Sentiment Breakdown (Donut) ────────────────────────────
df = spark.sql("""
    SELECT
        sentiment,
        COUNT(*) AS review_count
    FROM review_analytics
    WHERE sentiment IS NOT NULL
    GROUP BY sentiment
    ORDER BY review_count DESC
""").toPandas()

fig = px.pie(
    df,
    names="sentiment",
    values="review_count",
    title="😊 Overall Sentiment Breakdown",
    hole=0.4,
    color="sentiment",
    color_discrete_map={
        "Positive": "#2ECC71",
        "Neutral":  "#F39C12",
        "Negative": "#E74C3C"
    }
)
fig.update_traces(textinfo="percent+label+value")
fig.show()

In [ ]:
# ── Cell 6: Chart 2 — Avg Rating by Category (Bar) ───────────────────────────
df = spark.sql("""
    SELECT
        category,
        ROUND(AVG(rating), 2)    AS avg_rating,
        COUNT(review_id)         AS total_reviews
    FROM review_analytics
    GROUP BY category
    ORDER BY avg_rating DESC
""").toPandas()

fig = px.bar(
    df,
    x="category",
    y="avg_rating",
    title="⭐ Average Rating by Category",
    color="avg_rating",
    color_continuous_scale="YlGn",
    text="avg_rating",
    hover_data=["total_reviews"]
)
fig.update_traces(textposition="outside")
fig.update_layout(
    xaxis_title="Category",
    yaxis_title="Avg Rating",
    yaxis=dict(range=[0, 5])
)
fig.show()

In [ ]:
# ── Cell 7: Chart 3 — Sentiment by Category (Stacked Bar) ────────────────────
df = spark.sql("""
    SELECT
        category,
        SUM(CASE WHEN sentiment = 'Positive' THEN 1 ELSE 0 END) AS positive,
        SUM(CASE WHEN sentiment = 'Neutral'  THEN 1 ELSE 0 END) AS neutral,
        SUM(CASE WHEN sentiment = 'Negative' THEN 1 ELSE 0 END) AS negative
    FROM review_analytics
    GROUP BY category
    ORDER BY category
""").toPandas()

fig = go.Figure()
fig.add_trace(go.Bar(name="Positive", x=df["category"], y=df["positive"], marker_color="#2ECC71"))
fig.add_trace(go.Bar(name="Neutral",  x=df["category"], y=df["neutral"],  marker_color="#F39C12"))
fig.add_trace(go.Bar(name="Negative", x=df["category"], y=df["negative"], marker_color="#E74C3C"))
fig.update_layout(
    barmode="stack",
    title="📊 Sentiment Distribution by Category",
    xaxis_title="Category",
    yaxis_title="Number of Reviews",
    legend_title="Sentiment"
)
fig.show()

In [ ]:
# ── Cell 8: Chart 4 — Star Rating Distribution (Bar) ─────────────────────────
df = spark.sql("""
    SELECT
        rating,
        COUNT(*) AS review_count
    FROM review_analytics
    GROUP BY rating
    ORDER BY rating DESC
""").toPandas()

df["rating"] = df["rating"].astype(str) + " ⭐"

fig = px.bar(
    df,
    x="rating",
    y="review_count",
    title="🌟 Star Rating Distribution",
    color="review_count",
    color_continuous_scale="YlGn",
    text="review_count"
)
fig.update_traces(textposition="outside")
fig.update_layout(
    xaxis_title="Star Rating",
    yaxis_title="Number of Reviews"
)
fig.show()

In [ ]:
# ── Cell 9: Chart 5 — NPS Category Breakdown (Donut) ─────────────────────────
df = spark.sql("""
    SELECT
        nps_category,
        COUNT(*) AS count
    FROM review_analytics
    WHERE nps_category IS NOT NULL
    GROUP BY nps_category
    ORDER BY count DESC
""").toPandas()

fig = px.pie(
    df,
    names="nps_category",
    values="count",
    title="📣 NPS Category — Promoters vs Passives vs Detractors",
    hole=0.4,
    color="nps_category",
    color_discrete_map={
        "Promoter":  "#2ECC71",
        "Passive":   "#F39C12",
        "Detractor": "#E74C3C"
    }
)
fig.update_traces(textinfo="percent+label+value")
fig.show()

In [ ]:
# ── Cell 10: Chart 6 — Avg Rating by Customer Segment (Bar) ──────────────────
df = spark.sql("""
    SELECT
        customer_segment,
        ROUND(AVG(rating), 2) AS avg_rating,
        COUNT(review_id)      AS total_reviews
    FROM review_analytics
    WHERE customer_segment IS NOT NULL
    GROUP BY customer_segment
    ORDER BY avg_rating DESC
""").toPandas()

fig = px.bar(
    df,
    x="customer_segment",
    y="avg_rating",
    title="👥 Avg Rating by Customer Segment",
    color="customer_segment",
    color_discrete_sequence=px.colors.qualitative.Set2,
    text="avg_rating",
    hover_data=["total_reviews"]
)
fig.update_traces(textposition="outside")
fig.update_layout(
    showlegend=False,
    xaxis_title="Customer Segment",
    yaxis_title="Avg Rating",
    yaxis=dict(range=[0, 5])
)
fig.show()

In [ ]:
# ── Cell 11: Chart 7 — Monthly Review Volume Over Time (Line) ─────────────────
df = spark.sql("""
    SELECT
        review_year,
        review_month,
        CONCAT(CAST(review_year AS STRING), '-',
               LPAD(CAST(review_month AS STRING), 2, '0')) AS year_month,
        COUNT(review_id)      AS total_reviews,
        ROUND(AVG(rating), 2) AS avg_rating
    FROM review_analytics
    GROUP BY review_year, review_month
    ORDER BY review_year, review_month
""").toPandas()

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=("Monthly Review Volume", "Monthly Avg Rating"),
    shared_xaxes=True
)
fig.add_trace(
    go.Scatter(
        x=df["year_month"], y=df["total_reviews"],
        mode="lines+markers", name="Reviews",
        line=dict(color="#636EFA", width=2),
        fill="tozeroy"
    ),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(
        x=df["year_month"], y=df["avg_rating"],
        mode="lines+markers", name="Avg Rating",
        line=dict(color="#F39C12", width=2)
    ),
    row=2, col=1
)
fig.update_layout(
    title_text="📅 Monthly Review Volume and Avg Rating Over Time",
    showlegend=False,
    height=500
)
fig.show()

In [ ]:
# ── Cell 12: Chart 8 — Top 15 Products by Review Count (Horizontal Bar) ───────
df = spark.sql("""
    SELECT
        product_name,
        category,
        COUNT(review_id)      AS total_reviews,
        ROUND(AVG(rating), 2) AS avg_rating
    FROM review_analytics
    GROUP BY product_name, category
    ORDER BY total_reviews DESC
    LIMIT 15
""").toPandas()

fig = px.bar(
    df,
    x="total_reviews",
    y="product_name",
    orientation="h",
    title="🏆 Top 15 Most Reviewed Products",
    color="avg_rating",
    color_continuous_scale="YlGn",
    text="total_reviews",
    hover_data=["avg_rating", "category"]
)
fig.update_traces(textposition="outside")
fig.update_layout(
    yaxis={"categoryorder": "total ascending"},
    xaxis_title="Number of Reviews",
    yaxis_title="Product"
)
fig.show()

In [ ]:
# ── Cell 13: Chart 9 — Verified vs Unverified Reviews by Category (Grouped Bar)
df = spark.sql("""
    SELECT
        category,
        SUM(CASE WHEN verified_purchase = true  THEN 1 ELSE 0 END) AS verified,
        SUM(CASE WHEN verified_purchase = false THEN 1 ELSE 0 END) AS unverified
    FROM review_analytics
    GROUP BY category
    ORDER BY category
""").toPandas()

fig = go.Figure()
fig.add_trace(go.Bar(
    name="Verified",
    x=df["category"], y=df["verified"],
    marker_color="#2ECC71",
    text=df["verified"], textposition="outside"
))
fig.add_trace(go.Bar(
    name="Unverified",
    x=df["category"], y=df["unverified"],
    marker_color="#E74C3C",
    text=df["unverified"], textposition="outside"
))
fig.update_layout(
    barmode="group",
    title="✅ Verified vs Unverified Reviews by Category",
    xaxis_title="Category",
    yaxis_title="Number of Reviews",
    legend_title="Purchase Verification"
)
fig.show()

In [ ]:
# ── Cell 14: Chart 10 — Rating Trend (Improving/Declining/Stable) by Category
df = spark.sql("""
    SELECT
        category,
        rating_trend,
        COUNT(DISTINCT product_id) AS product_count
    FROM review_analytics
    WHERE rating_trend IS NOT NULL
    GROUP BY category, rating_trend
    ORDER BY category
""").toPandas()

fig = px.bar(
    df,
    x="category",
    y="product_count",
    color="rating_trend",
    barmode="group",
    title="📈 Rating Trend by Category (Improving / Stable / Declining)",
    color_discrete_map={
        "Improving": "#2ECC71",
        "Stable":    "#F39C12",
        "Declining": "#E74C3C"
    },
    text="product_count"
)
fig.update_traces(textposition="outside")
fig.update_layout(
    xaxis_title="Category",
    yaxis_title="Number of Products",
    legend_title="Rating Trend"
)
fig.show()

In [ ]:
# ── Cell 15: Chart 11 — Reviews by Country (Bar) ─────────────────────────────
df = spark.sql("""
    SELECT
        country,
        COUNT(review_id)      AS total_reviews,
        ROUND(AVG(rating), 2) AS avg_rating
    FROM review_analytics
    WHERE country IS NOT NULL
    GROUP BY country
    ORDER BY total_reviews DESC
""").toPandas()

fig = px.bar(
    df,
    x="country",
    y="total_reviews",
    title="🌍 Review Volume by Country",
    color="avg_rating",
    color_continuous_scale="Blues",
    text="total_reviews",
    hover_data=["avg_rating"]
)
fig.update_traces(textposition="outside")
fig.update_layout(
    xaxis_title="Country",
    yaxis_title="Total Reviews"
)
fig.show()

In [ ]:
# ── Cell 16: Chart 12 — Star Count Heatmap (Category x Star) ─────────────────
df = spark.sql("""
    SELECT
        category,
        SUM(star_5_count) AS star_5,
        SUM(star_4_count) AS star_4,
        SUM(star_3_count) AS star_3,
        SUM(star_2_count) AS star_2,
        SUM(star_1_count) AS star_1
    FROM review_analytics
    GROUP BY category
""").toPandas()

pivot = df.set_index("category")[["star_5","star_4","star_3","star_2","star_1"]]
pivot.columns = ["5 ⭐","4 ⭐","3 ⭐","2 ⭐","1 ⭐"]

fig = px.imshow(
    pivot,
    title="🔥 Star Rating Distribution Heatmap (Category × Stars)",
    color_continuous_scale="YlGn",
    text_auto=True,
    aspect="auto",
    labels=dict(x="Star Rating", y="Category", color="Count")
)
fig.update_layout(
    xaxis_title="Star Rating",
    yaxis_title="Category"
)
fig.show()